# 01 — Prompt Templates & LCEL Chains

LCEL (LangChain Expression Language) — the `|` pipe syntax — is **not deprecated**; it's still the recommended way to build simple, composable pipelines in v1.0. What *is* deprecated is `LLMChain`. If you see `LLMChain(llm=..., prompt=...)` in a tutorial, that's the old way — this notebook shows the current way.

**Run `00_setup_and_basics.ipynb` first** (or re-run the setup cell below) so `model` and `MODEL_ID` exist.


In [ ]:
import os
from getpass import getpass
from langchain.chat_models import init_chat_model

if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("Enter your OPENAI_API_KEY: ")
MODEL_ID = "openai:gpt-4.1-mini"

model = init_chat_model(MODEL_ID, temperature=0.3)

## 1. `ChatPromptTemplate` — reusable, parameterized prompts

Instead of f-string-ing your prompts by hand (which gets messy and unsafe once you accept user input), define a template once and fill in variables at call time.


In [ ]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are an expert {domain} reviewer. Be specific and concise."),
    ("human", "Review this: {content}"),
])

# .invoke() on a prompt template returns a formatted list of messages, not a model response yet
formatted = prompt.invoke({"domain": "Python code", "content": "def add(a,b): return a+b"})
print(formatted.to_messages())

## 2. The LCEL pipe: `prompt | model`

This is the core pattern. `|` composes runnables — anything with `.invoke()` — into a pipeline. LangChain handles passing the output of the left side into the input of the right side.


In [ ]:
chain = prompt | model

result = chain.invoke({"domain": "Python code", "content": "def add(a,b): return a+b"})
print(result.content)

## 3. Output parsers — get a plain string instead of an `AIMessage`

Add `StrOutputParser()` to the end of the chain when you just want text back, not the full message object.


In [ ]:
from langchain_core.output_parsers import StrOutputParser

chain = prompt | model | StrOutputParser()

result = chain.invoke({"domain": "Python code", "content": "def add(a,b): return a+b"})
print(type(result))
print(result)

## 4. Chaining multiple steps together

Because every piece is a "runnable," you can chain as many steps as you like — including plain Python functions via `RunnableLambda`.


In [ ]:
from langchain_core.runnables import RunnableLambda

summarize_prompt = ChatPromptTemplate.from_messages([
    ("system", "Summarize the input in exactly one sentence."),
    ("human", "{text}"),
])

uppercase = RunnableLambda(lambda text: text.upper())

pipeline = summarize_prompt | model | StrOutputParser() | uppercase

long_text = (
    "LangChain is a framework for building applications powered by language models. "
    "It provides abstractions for prompts, chains, tools, and agents, and is built on "
    "top of LangGraph for stateful orchestration."
)

print(pipeline.invoke({"text": long_text}))

## 5. Batching and parallelism

`.batch()` runs multiple inputs concurrently — much faster than looping `.invoke()` calls one at a time.


In [ ]:
inputs = [
    {"text": "Python is a general-purpose programming language known for readability."},
    {"text": "The Great Wall of China is over 13,000 miles long."},
    {"text": "Photosynthesis converts light energy into chemical energy in plants."},
]

results = (summarize_prompt | model | StrOutputParser()).batch(inputs)
for r in results:
    print("-", r)

## 6. Running steps in parallel with `RunnableParallel`

Useful when you want several independent things computed from the same input (e.g. a summary *and* a sentiment score at once).


In [ ]:
from langchain_core.runnables import RunnableParallel

sentiment_prompt = ChatPromptTemplate.from_messages([
    ("system", "Classify the sentiment as exactly one word: positive, negative, or neutral."),
    ("human", "{text}"),
])

parallel_chain = RunnableParallel(
    summary=summarize_prompt | model | StrOutputParser(),
    sentiment=sentiment_prompt | model | StrOutputParser(),
)

out = parallel_chain.invoke({"text": long_text})
print(out)

---
### Key takeaways
- `ChatPromptTemplate` for reusable, variable-filled prompts — safer and cleaner than f-strings.
- `prompt | model | parser` is the standard LCEL chain shape.
- `RunnableLambda` lets you drop plain Python functions into a chain.
- `.batch()` for concurrent processing, `RunnableParallel` for fan-out on a single input.
- `LLMChain` is deprecated — don't use it even if a tutorial shows it.

**Next:** `02_tools_and_agents.ipynb` — giving the model tools it can call.
